In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch", "sentence-transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])

In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/nli-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
sample_size = 300
device = "mps" if torch.backends.mps.is_available() else "cpu"
pipe_device = "mps" if device == "mps" else -1
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "sample_size": sample_size,
    "device": device,
    "pipeline_device": pipe_device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

n = min(sample_size, len(df))
df = df.sample(n=n, random_state=seed).reset_index(drop=True)

print({"num_examples_sampled": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipe_device,
    return_all_scores=True,
)

print({
    "pipeline_task": clf.task,
    "model_name": model_name,
    "pipeline_device": pipe_device,
})

In [ ]:
pairs = [
    {"text": s1, "text_pair": s2}
    for s1, s2 in zip(df["sentence1"].tolist(), df["sentence2"].tolist())
]

outputs = clf(
    pairs,
    batch_size=batch_size,
    truncation=True,
)

label_order = None
if hasattr(clf.model, "config") and hasattr(clf.model.config, "id2label"):
    id2label = clf.model.config.id2label
    label_order = [id2label[i] for i in sorted(id2label.keys())]

def scores_to_map(item):
    return {str(x["label"]).lower(): float(x["score"]) for x in item}

score_maps = [scores_to_map(x) for x in outputs]

contradiction = np.array([m.get("contradiction", np.nan) for m in score_maps], dtype=np.float32)
neutral = np.array([m.get("neutral", np.nan) for m in score_maps], dtype=np.float32)
entailment = np.array([m.get("entailment", np.nan) for m in score_maps], dtype=np.float32)

predicted_score_0_5 = (entailment * 5.0).astype(np.float32)
predicted_score_rounded = np.rint(predicted_score_0_5).clip(0, 5).astype(np.int32)
label_rounded = np.rint(df["label"].to_numpy(dtype=np.float32)).clip(0, 5).astype(np.int32)

predicted_label_name = [max(x, key=lambda y: y["score"])["label"] for x in outputs]

preview = pd.DataFrame({
    "top_label": predicted_label_name[:10],
    "contradiction": contradiction[:10],
    "neutral": neutral[:10],
    "entailment": entailment[:10],
    "predicted_score_0_5": predicted_score_0_5[:10],
    "predicted_score_rounded": predicted_score_rounded[:10],
    "label": df["label"].head(10).to_numpy(),
    "label_rounded": label_rounded[:10],
})
print({"label_order": label_order})
print(preview)

In [ ]:
labels = df["label"].to_numpy(dtype=np.float32)

pearson_pred = pearsonr(predicted_score_0_5, labels).statistic
spearman_pred = spearmanr(predicted_score_0_5, labels).statistic
mae_pred = np.mean(np.abs(predicted_score_0_5 - labels))

rounded_exact_match = float(np.mean(predicted_score_rounded == label_rounded))
rounded_within_1 = float(np.mean(np.abs(predicted_score_rounded - label_rounded) <= 1))
mean_abs_round_diff = float(np.mean(np.abs(predicted_score_rounded - label_rounded)))

results_df = df.copy()
results_df["contradiction_prob"] = contradiction
results_df["neutral_prob"] = neutral
results_df["entailment_prob"] = entailment
results_df["top_label"] = predicted_label_name
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["predicted_score_rounded"] = predicted_score_rounded
results_df["label_rounded"] = label_rounded
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["rounded_match"] = results_df["predicted_score_rounded"] == results_df["label_rounded"]
results_df["rounded_abs_diff"] = np.abs(results_df["predicted_score_rounded"] - results_df["label_rounded"])

print(results_df[[
    "sentence1", "sentence2", "label", "label_rounded", "top_label",
    "entailment_prob", "predicted_score_0_5", "predicted_score_rounded",
    "abs_error", "rounded_match", "rounded_abs_diff"
]].head(10))

In [ ]:
agreement_table = pd.crosstab(
    results_df["label_rounded"],
    results_df["predicted_score_rounded"],
    rownames=["label_rounded"],
    colnames=["predicted_score_rounded"],
    dropna=False,
)

worst_examples = results_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5",
    "label_rounded", "predicted_score_rounded", "abs_error", "rounded_abs_diff"
]].sort_values(["abs_error", "rounded_abs_diff"], ascending=False).head(10)

runtime_seconds = time.time() - start_time

print("rounded_agreement_table")
print(agreement_table.to_string())
print()
print("worst_examples")
print(worst_examples.to_string(index=False))
print()
print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples_sampled: {len(df)}")
print(f"pearson_predicted_score: {pearson_pred:.6f}")
print(f"spearman_predicted_score: {spearman_pred:.6f}")
print(f"mae_predicted_score: {mae_pred:.6f}")
print(f"rounded_exact_match: {rounded_exact_match:.6f}")
print(f"rounded_within_1: {rounded_within_1:.6f}")
print(f"mean_abs_round_diff: {mean_abs_round_diff:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")